In [1]:
import os
import duckdb
import pandas as pd
from sec_edgar_downloader import Downloader
import warnings
warnings.filterwarnings('ignore')

# Create folders
os.makedirs('../data/raw/sec_filings', exist_ok=True)
os.makedirs('../data/processed/sec_text', exist_ok=True)

print("Ready!")

Ready!


In [3]:
from sec_edgar_downloader import Downloader

# New API - save_path is NOT passed in constructor
dl = Downloader(
    company_name="Munta Research",
    email_address="your_email@gmail.com"
)

# Our same 10 companies
tickers = ['AAPL', 'MSFT', 'JPM', 'BAC', 'GS', 'AMZN', 'TSLA', 'XOM', 'JNJ', 'WMT']

# Download last 3 years of 10-K filings per company
print("Downloading SEC 10-K filings... this may take 5-10 minutes")

for ticker in tickers:
    try:
        dl.get("10-K", ticker, limit=3)
        print(f"✅ {ticker} done")
    except Exception as e:
        print(f"❌ {ticker} failed: {e}")

print("\nAll downloads complete!")

✅ AAPL done
✅ MSFT done
✅ JPM done
✅ BAC done
✅ GS done
✅ AMZN done
✅ TSLA done
✅ XOM done
✅ JNJ done
✅ WMT done

All downloads complete!


In [11]:
import re
import os
import duckdb
import pandas as pd

def extract_text_from_filing(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    clean = re.sub(r'<[^>]+>', ' ', content)
    clean = re.sub(r'\s+', ' ', clean).strip()
    clean = ' '.join([w for w in clean.split() if len(w) > 1])
    return clean[:50000]

# Correct absolute path
base_path = "C:/Users/munta/Documents/financial-risk-copilot/sec-edgar-filings"
tickers = ['AAPL', 'MSFT', 'JPM', 'BAC', 'GS', 'AMZN', 'TSLA', 'XOM', 'JNJ', 'WMT']
records = []

for ticker in tickers:
    filing_path = os.path.join(base_path, ticker, "10-K")
    
    if not os.path.exists(filing_path):
        print(f"⚠️ No filings found for {ticker}")
        continue
    
    filing_folders = sorted(os.listdir(filing_path), reverse=True)
    
    for folder in filing_folders[:3]:
        folder_path = os.path.join(filing_path, folder)
        
        for filename in os.listdir(folder_path):
            if filename.endswith('.htm') or filename.endswith('.txt'):
                full_path = os.path.join(folder_path, filename)
                text = extract_text_from_filing(full_path)
                
                if len(text) > 1000:
                    records.append({
                        'ticker': ticker,
                        'filing_date': folder,
                        'filename': filename,
                        'text': text,
                        'char_count': len(text)
                    })
                    print(f"✅ {ticker} | {folder} | {len(text):,} chars")
                break

print(f"\nTotal filings extracted: {len(records)}")

✅ AAPL | 0000320193-25-000079 | 50,000 chars
✅ AAPL | 0000320193-24-000123 | 50,000 chars
✅ AAPL | 0000320193-23-000106 | 50,000 chars
✅ MSFT | 0000950170-25-100235 | 50,000 chars
✅ MSFT | 0000950170-24-087843 | 50,000 chars
✅ MSFT | 0000950170-23-035122 | 50,000 chars
✅ JPM | 0001628280-26-008131 | 50,000 chars
✅ JPM | 0000019617-25-000270 | 50,000 chars
✅ JPM | 0000019617-24-000225 | 50,000 chars
✅ BAC | 0000070858-26-000157 | 50,000 chars
✅ BAC | 0000070858-25-000139 | 50,000 chars
✅ BAC | 0000070858-24-000122 | 50,000 chars
✅ GS | 0000886982-26-000091 | 50,000 chars
✅ GS | 0000886982-25-000005 | 50,000 chars
✅ GS | 0000886982-24-000006 | 50,000 chars
✅ AMZN | 0001018724-26-000004 | 50,000 chars
✅ AMZN | 0001018724-25-000004 | 50,000 chars
✅ AMZN | 0001018724-24-000008 | 50,000 chars
✅ TSLA | 0001628280-26-003952 | 50,000 chars
✅ TSLA | 0001628280-25-003063 | 50,000 chars
✅ TSLA | 0001628280-24-002390 | 50,000 chars
✅ XOM | 0000034088-26-000045 | 50,000 chars
✅ XOM | 0000034088-25-0

In [12]:
df_filings = pd.DataFrame(records)
print(df_filings[['ticker', 'filing_date', 'char_count']])

con = duckdb.connect('C:/Users/munta/Documents/financial-risk-copilot/data/financial_warehouse.duckdb')

con.execute("DROP TABLE IF EXISTS sec_filings")
con.execute("CREATE TABLE sec_filings AS SELECT * FROM df_filings")

result = con.execute("""
    SELECT ticker, COUNT(*) as filings, AVG(char_count) as avg_chars
    FROM sec_filings 
    GROUP BY ticker
    ORDER BY ticker
""").df()

print("\nFilings in warehouse:")
print(result)
con.close()

   ticker           filing_date  char_count
0    AAPL  0000320193-25-000079       50000
1    AAPL  0000320193-24-000123       50000
2    AAPL  0000320193-23-000106       50000
3    MSFT  0000950170-25-100235       50000
4    MSFT  0000950170-24-087843       50000
5    MSFT  0000950170-23-035122       50000
6     JPM  0001628280-26-008131       50000
7     JPM  0000019617-25-000270       50000
8     JPM  0000019617-24-000225       50000
9     BAC  0000070858-26-000157       50000
10    BAC  0000070858-25-000139       50000
11    BAC  0000070858-24-000122       50000
12     GS  0000886982-26-000091       50000
13     GS  0000886982-25-000005       50000
14     GS  0000886982-24-000006       50000
15   AMZN  0001018724-26-000004       50000
16   AMZN  0001018724-25-000004       50000
17   AMZN  0001018724-24-000008       50000
18   TSLA  0001628280-26-003952       50000
19   TSLA  0001628280-25-003063       50000
20   TSLA  0001628280-24-002390       50000
21    XOM  0000034088-26-000045 

In [13]:
os.makedirs('C:/Users/munta/Documents/financial-risk-copilot/data/processed/sec_text', exist_ok=True)

for _, row in df_filings.iterrows():
    filename = f"C:/Users/munta/Documents/financial-risk-copilot/data/processed/sec_text/{row['ticker']}_{row['filing_date']}.txt"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"Company: {row['ticker']}\n")
        f.write(f"Filing Date: {row['filing_date']}\n")
        f.write(f"{'='*50}\n\n")
        f.write(row['text'])

print(f"✅ Saved {len(df_filings)} text files")
print(os.listdir('C:/Users/munta/Documents/financial-risk-copilot/data/processed/sec_text/')[:5])

✅ Saved 30 text files
['AAPL_0000320193-23-000106.txt', 'AAPL_0000320193-24-000123.txt', 'AAPL_0000320193-25-000079.txt', 'AMZN_0001018724-24-000008.txt', 'AMZN_0001018724-25-000004.txt']
